# Train Rook on a free Kaggle GPU (runs in the background)

Same rules as the GitHub pipeline: random-initialized weights, full-sequence next-token prediction on documented text, neutral `Person A` / `Person B` labels, no identity text, no assistant training, BPE tokenizer learned from Rook's own training text.

**Before you start (right-hand Settings panel):** Accelerator → **GPU T4 x2** or **GPU P100**; Internet → **On**. Then press **Save Version → Save & Run All (Commit)**. You can close the page; the run continues on Kaggle's computers.

The trained files appear under the notebook's **Output** tab when the run finishes.

## 1. Settings

In [ ]:
MAX_MINUTES = 480       # training time in this run (Kaggle stops a run after about 9-12 hours; lower it if yours stops early)
BATCH = 32              # windows per step (lower to 16 if you see an out-of-memory error)
STORIES = 500000        # TinyStories to use (more text = better; about 700 bytes each)
REPO = "https://github.com/meshalalessa321-boop/Rook-app"
SAVE_DIR = "/kaggle/working/rook-gpu"   # everything here is kept as the notebook's Output


## 2. Check the GPU

In [ ]:
import torch, os
assert torch.cuda.is_available(), "No GPU. Open Settings on the right, set Accelerator to a GPU, then run again."
print("GPU:", torch.cuda.get_device_name(0))
os.makedirs(SAVE_DIR, exist_ok=True)


## 3. Get Rook's code and build the documented text corpus

In [ ]:
%%bash
set -e
cd /kaggle/working
rm -rf rook
git clone --depth 1 https://github.com/meshalalessa321-boop/Rook-app rook
pip install -q datasets onnx onnxruntime


In [ ]:
import subprocess, sys
def run(cmd):
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True, cwd="/kaggle/working/rook")
run([sys.executable, "training/prepare_wikitext.py"])
run([sys.executable, "training/prepare_tinystories.py", "--max-stories", str(STORIES)])
run([sys.executable, "training/prepare_dialogue.py"])
run([sys.executable, "training/rook_brain.py", "--audit-only", "--out-dir", SAVE_DIR])


## 4. Train

Every step predicts the next token across all of the text. Nothing tells Rook what he is. The loss printed is on held-out text; it should fall over time. Training stops by itself after `MAX_MINUTES`, with the learning rate easing down over that time.

In [ ]:
args = [sys.executable, "training/rook_brain.py", "--out-dir", SAVE_DIR, "--fresh",
        "--steps", "1000000", "--max-minutes", str(MAX_MINUTES), "--run-name", "kaggle-gpu",
        "--ctx", "256", "--batch", str(BATCH), "--accum", "1", "--lr", "6e-4", "--warmup", "500"]
run(args)


## 5. Look at what he writes (from the best save)

In [ ]:
import json, sys
sys.path.insert(0, "/kaggle/working/rook/training")
import torch
from train_rook_next import Rook
from rook_tokenizer import ByteBPE, byte_tokenizer

ck = torch.load(f"{SAVE_DIR}/best.pt", map_location="cuda", weights_only=False)
tok = ByteBPE.from_json(ck["tokenizer"]) if ck.get("tokenizer") else byte_tokenizer()
cfg = ck["config"]
model = Rook(cfg["d"], cfg["layers"], cfg["heads"], cfg["ctx"], vocab=tok.vocab_size).cuda()
model.load_state_dict(ck["model"]); model.eval()
print("trained steps:", ck["step"], "| held-out loss:", round(ck["val_loss"], 3))

@torch.no_grad()
def talk(text, new_tokens=80, temperature=0.7, top_k=32):
    ids = [tok.bos] + tok.encode(f"Person A: {text}\nPerson B:")
    for _ in range(new_tokens):
        window = ids[-cfg["ctx"]:]
        logits = model(torch.tensor([window], device="cuda"))[0, len(window) - 1]
        logits[tok.pad] = logits[tok.bos] = -1e9
        top = torch.topk(logits, top_k)
        probs = torch.softmax(top.values / temperature, dim=0)
        nxt = int(top.indices[torch.multinomial(probs, 1)])
        if nxt == tok.eos: break
        ids.append(nxt)
        if "\nPerson A:" in tok.decode(ids): break
    reply = tok.decode(ids[1 + len(tok.encode(f'Person A: {text}\nPerson B:')):])
    return reply.split("\nPerson A:")[0].strip()

for prompt in ["Hello! How are you today?", "What do you like to do?", "I had pizza for dinner.", "Tell me about your day."]:
    print("Person A:", prompt); print("Person B:", talk(prompt)); print()


## 6. Export for the chat page

This writes the ONNX model, its description and the tokenizer into the `export/` folder of the Output tab. Nothing is published to the chat page from here.

In [ ]:
import pathlib
exp = pathlib.Path(SAVE_DIR) / "export"; exp.mkdir(exist_ok=True)
run([sys.executable, "training/export_rook_brain_onnx.py", "--checkpoint", f"{SAVE_DIR}/best.pt",
     "--model-out", str(exp / "rook-brain.onnx"), "--manifest-out", str(exp / "brain.json"),
     "--tokenizer-out", str(exp / "tokenizer.json")])
print("Exported to", exp, os.listdir(exp))
